# Mitra Classifier — End-to-End Classification with Your Own Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-classifier-pipeline/blob/main/tutorials/mitra_classifier_colab.ipynb)

Use the Mitra Classifier weights distributed through the DIMER Model Repository, or the exact pinned upstream checkpoint as a fallback. If you do not yet have a labelled CSV, use the bundled FreshRetailNet sample.

**No DIMER Workbench access is required.** Data is processed in Google Colab, not by DIMER. Do not upload confidential, sensitive, or restricted data unless that environment is permitted.


## 1. Install the runtime

The `mitra` extra supplies the Mitra runtime. PyTorch is left to Colab so its CUDA build stays compatible with the selected accelerator.


In [ ]:
%pip install -q "autogluon.tabular[mitra]==1.5.0"


## 2. Acquire and verify the checkpoint

- **DIMER ZIP** — upload the DIMER ZIP containing `model.safetensors`; the notebook retrieves the matching pinned `config.json`.
- **Pinned upstream** — retrieve both files from the exact pinned AutoGluon revision.

Both files are SHA-256 verified and installed into an isolated Hugging Face cache before AutoGluon is put offline. Network requests use a finite timeout so transient outages fail clearly instead of hanging indefinitely.


In [ ]:
import hashlib, json, os, random, shutil, urllib.request, zipfile
from pathlib import Path

MODEL_ID = "autogluon/mitra-classifier"
PINNED_REVISION = "c425e9fa0910a6be1c494321792e7ba2a1367b1a"
EXPECTED_WEIGHTS_SHA256 = "e06a055e91a3baeffc37f9cf634d9e69a27d904b6686131dc3b702f9c0126b19"
EXPECTED_CONFIG_SHA256 = "2c96c24dd25f64e92753f6f2ba00cc7833b9923459403dcd8504e8700c0995df"
NETWORK_TIMEOUT_SECONDS = 30

HF_HOME = Path("/content/mitra-hf")
MODEL_DIR = Path("/content/mitra-model")
HF_HOME.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def fetch_pinned(name, dest):
    url = f"https://huggingface.co/{MODEL_ID}/resolve/{PINNED_REVISION}/{name}?download=true"
    print("Retrieving pinned", name)
    with urllib.request.urlopen(url, timeout=NETWORK_TIMEOUT_SECONDS) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)

def verify(path, expected, label):
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f"{label} checksum mismatch.\nExpected: {expected}\nActual:   {actual}")
    print(f"✓ {label} verified: {actual[:12]}…")

def weights_from_dimer(dest):
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one DIMER ZIP or model.safetensors.")
    p = Path("/content") / next(iter(uploaded))
    if p.suffix.lower() == ".safetensors":
        shutil.copy2(p, dest)
        return
    if p.suffix.lower() != ".zip":
        raise ValueError("Expected a DIMER ZIP or model.safetensors.")
    with zipfile.ZipFile(p) as z:
        matches = [i for i in z.infolist() if not i.is_dir() and Path(i.filename).name == "model.safetensors"]
        if len(matches) != 1:
            raise RuntimeError(f"Expected one model.safetensors in the DIMER ZIP; found {len(matches)}.")
        with z.open(matches[0]) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)

def install_offline_snapshot(weights, config):
    snapshot = sha256_file(weights)[:40]
    repo = HF_HOME / "hub" / ("models--" + MODEL_ID.replace("/", "--"))
    snap = repo / "snapshots" / snapshot
    refs = repo / "refs"
    snap.mkdir(parents=True, exist_ok=True)
    refs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights, snap / "model.safetensors")
    shutil.copy2(config, snap / "config.json")
    (refs / "main").write_text(snapshot)
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    return snap

MODEL_SOURCE = "Pinned upstream"  # @param ["DIMER ZIP", "Pinned upstream"]

weights_path = MODEL_DIR / "model.safetensors"
config_path = MODEL_DIR / "config.json"

if MODEL_SOURCE == "DIMER ZIP":
    weights_from_dimer(weights_path)
    fetch_pinned("config.json", config_path)
else:
    fetch_pinned("model.safetensors", weights_path)
    fetch_pinned("config.json", config_path)

verify(weights_path, EXPECTED_WEIGHTS_SHA256, "model.safetensors")
verify(config_path, EXPECTED_CONFIG_SHA256, "config.json")
print("✓ Offline snapshot:", install_offline_snapshot(weights_path, config_path))


## 3. Choose a dataset

If you do not have a dataset, choose **Sample dataset (FreshRetailNet)**. The bundled ZIP contains `train.csv`, `val.csv`, and `test.csv`; the notebook preserves those provided partitions instead of randomly re-splitting them.

The sample has 4,180 training rows, 1,600 validation rows, 1,600 test rows, 17 features, and a 3-class demand-band target. It is derived from FreshRetailNet-50K and redistributed under **CC BY 4.0** for tutorial/smoke-test use, not benchmarking.

[Read the sample DATASET_CARD.md](https://github.com/kurtvalcorza/mitra-classifier-pipeline/blob/main/examples/sample-data/DATASET_CARD.md)

For **Upload CSV**, provide one labelled CSV and set the target column. The notebook creates a stratified holdout.


In [ ]:
import io
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_SOURCE = "Sample dataset (FreshRetailNet)"  # @param ["Sample dataset (FreshRetailNet)", "Upload CSV"]
TARGET_COLUMN = "target"                         # @param {type:"string"}
DROP_COLUMNS = ""                                # @param {type:"string"}
VALIDATION_SPLIT = 0.20                          # @param {type:"number"}
SEED = 42                                        # @param {type:"integer"}

SAMPLE_ZIP_URL = "https://raw.githubusercontent.com/kurtvalcorza/mitra-classifier-pipeline/main/examples/sample-data/freshretailnet-band-h7.zip"
SAMPLE_CARD_URL = "https://github.com/kurtvalcorza/mitra-classifier-pipeline/blob/main/examples/sample-data/DATASET_CARD.md"

sample_test_data = None
using_presplit_sample = DATA_SOURCE == "Sample dataset (FreshRetailNet)"

if using_presplit_sample:
    with urllib.request.urlopen(SAMPLE_ZIP_URL, timeout=NETWORK_TIMEOUT_SECONDS) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        names = {Path(n).name: n for n in z.namelist() if not n.endswith("/")}
        required = {"train.csv", "val.csv", "test.csv"}
        missing = sorted(required - set(names))
        if missing:
            raise RuntimeError(f"Sample ZIP missing: {missing}")
        train_data = pd.read_csv(z.open(names["train.csv"]))
        holdout_data = pd.read_csv(z.open(names["val.csv"]))
        sample_test_data = pd.read_csv(z.open(names["test.csv"]))
    TARGET_COLUMN = "target"
    print("✓ Using FreshRetailNet sample with preserved train/val/test splits.")
    print("  Dataset card:", SAMPLE_CARD_URL)
else:
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".csv")]
    if len(names) != 1:
        raise RuntimeError("Upload exactly one labelled CSV.")
    data = pd.read_csv(Path("/content") / names[0])

drop_columns = [c.strip() for c in DROP_COLUMNS.split(",") if c.strip() and c.strip() != TARGET_COLUMN]

def prepare(df, name):
    if df.columns.duplicated().any():
        raise ValueError(f"{name}: duplicate column names are not supported.")
    if TARGET_COLUMN not in df.columns:
        raise ValueError(f"{name}: target {TARGET_COLUMN!r} not found.")
    out = df.drop(columns=[c for c in drop_columns if c in df.columns], errors="ignore").dropna(subset=[TARGET_COLUMN]).copy()
    features = [c for c in out.columns if c != TARGET_COLUMN]
    counts = out[TARGET_COLUMN].value_counts()
    errors = []
    if len(out) < 50:
        errors.append("use at least 50 labelled rows")
    if not features:
        errors.append("no feature columns remain")
    if len(features) > 500:
        errors.append(f"{len(features)} features exceed the 500-feature limit")
    if not 2 <= len(counts) <= 10:
        errors.append(f"target has {len(counts)} classes; Mitra requires 2–10")
    if counts.empty or counts.min() < 2:
        errors.append("every class needs at least 2 rows")
    if errors:
        raise ValueError(f"{name} is not ready: " + "; ".join(errors))
    return out, features

if using_presplit_sample:
    train_data, features = prepare(train_data, "train.csv")
    holdout_data, val_features = prepare(holdout_data, "val.csv")
    sample_test_data, test_features = prepare(sample_test_data, "test.csv")
    if features != val_features or features != test_features:
        raise ValueError("Sample train/val/test feature columns do not match.")
else:
    clean, features = prepare(data, "uploaded CSV")
    if not 0.05 <= VALIDATION_SPLIT <= 0.40:
        raise ValueError("VALIDATION_SPLIT must be 0.05–0.40.")
    train_data, holdout_data = train_test_split(
        clean,
        test_size=VALIDATION_SPLIT,
        random_state=SEED,
        stratify=clean[TARGET_COLUMN],
    )
    if len(train_data) > 10_000:
        train_data, _ = train_test_split(
            train_data,
            train_size=10_000,
            random_state=SEED,
            stratify=train_data[TARGET_COLUMN],
        )

FEATURE_COLUMNS = [c for c in train_data.columns if c != TARGET_COLUMN]
NUM_CLASSES = train_data[TARGET_COLUMN].nunique()
PROBLEM_TYPE = "binary" if NUM_CLASSES == 2 else "multiclass"

display(pd.DataFrame({
    "Item": ["Training rows", "Holdout rows", "Features", "Target", "Classes"],
    "Value": [len(train_data), len(holdout_data), len(FEATURE_COLUMNS), TARGET_COLUMN, NUM_CLASSES],
}))
display(train_data[TARGET_COLUMN].value_counts().rename("training rows").to_frame())
if sample_test_data is not None:
    print(f"✓ Independent sample test rows: {len(sample_test_data):,}")
if len(train_data) > 5_000:
    print("⚠ Above Mitra's particularly strong reported ≤5,000-sample regime.")
if len(FEATURE_COLUMNS) > 100:
    print("⚠ Above Mitra's particularly strong reported ≤100-feature regime.")


## 4. Evaluate pretrained Mitra, then optionally fine-tune

`fine_tune=False` uses labelled examples as context without updating weights. Fine-tuning requires a GPU. The bundled sample's `val.csv` is the holdout and `test.csv` is also reported as an independent test set.

**Colab memory note:** standard Colab can sit near AutoGluon's memory guard. `MAX_MEMORY_USAGE_RATIO=1.10` slightly relaxes the guard; values above 1.0 increase OOM risk.

**Metric note:** AutoGluon reports lower-is-better metrics such as `log_loss` with their sign flipped so that every score is higher-is-better. The helper below converts `log_loss` back to the conventional positive loss value before display/export.


In [ ]:
import torch
from autogluon.tabular import TabularPredictor

EVAL_METRIC = "accuracy"          # @param ["accuracy", "balanced_accuracy", "log_loss", "f1_macro", "mcc"]
BASELINE_TIME_LIMIT = 300         # @param {type:"integer"}
RUN_FINE_TUNING = False           # @param {type:"boolean"}
FINE_TUNE_STEPS = 0               # @param {type:"integer"}
FINE_TUNE_TIME_LIMIT = 600        # @param {type:"integer"}
MAX_MEMORY_USAGE_RATIO = 1.10     # @param {type:"number"}

CUDA_AVAILABLE = torch.cuda.is_available()
print("CUDA available:", CUDA_AVAILABLE, torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "")

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def fit_mitra(fine_tune, path, time_limit, steps=0):
    seed_everything()
    hp = {"fine_tune": fine_tune, "seed": SEED}
    if EVAL_METRIC in {"accuracy", "log_loss"}:
        hp["metric"] = EVAL_METRIC
    if fine_tune and steps > 0:
        hp["fine_tune_steps"] = steps
    predictor = TabularPredictor(
        label=TARGET_COLUMN,
        problem_type=PROBLEM_TYPE,
        eval_metric=EVAL_METRIC,
        path=path,
        verbosity=2,
    )
    predictor.fit(
        train_data,
        hyperparameters={"MITRA": hp},
        fit_weighted_ensemble=False,
        time_limit=time_limit,
        ag_args_fit={"max_memory_usage_ratio": MAX_MEMORY_USAGE_RATIO},
    )
    if not any("mitra" in n.lower() for n in predictor.model_names()):
        raise RuntimeError(f"Expected Mitra; AutoGluon trained {predictor.model_names()}.")
    return predictor

def metrics(predictor, frame):
    raw = predictor.evaluate(frame, auxiliary_metrics=True, silent=True)
    # AutoGluon evaluate() exposes lower-is-better metrics in higher-is-better form.
    # For log_loss specifically, convert the negative score back to conventional positive loss.
    return {k: float(-v if k == "log_loss" else v) for k, v in raw.items()}

baseline_predictor = fit_mitra(False, "/content/mitra-baseline", BASELINE_TIME_LIMIT)
baseline_metrics = metrics(baseline_predictor, holdout_data)
display(pd.Series(baseline_metrics, name="Pretrained — holdout").to_frame())

if sample_test_data is not None:
    display(pd.Series(
        metrics(baseline_predictor, sample_test_data),
        name="Pretrained — sample test",
    ).to_frame())

finetuned_predictor = finetuned_metrics = None
if RUN_FINE_TUNING:
    if not CUDA_AVAILABLE:
        raise RuntimeError("Fine-tuning requires a GPU. Choose Runtime → Change runtime type → GPU.")
    finetuned_predictor = fit_mitra(
        True,
        "/content/mitra-finetuned",
        FINE_TUNE_TIME_LIMIT,
        FINE_TUNE_STEPS,
    )
    finetuned_metrics = metrics(finetuned_predictor, holdout_data)
    display(pd.DataFrame({"Pretrained": baseline_metrics, "Fine-tuned": finetuned_metrics}))
    if sample_test_data is not None:
        display(pd.Series(
            metrics(finetuned_predictor, sample_test_data),
            name="Fine-tuned — sample test",
        ).to_frame())
else:
    print("Fine-tuning skipped. Set RUN_FINE_TUNING=True on a GPU to run it.")


## 5. Classify new rows

Upload an unlabelled CSV with the same feature columns. For the bundled sample, `test.csv` was already evaluated above.


In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}

if RUN_NEW_DATA_INFERENCE:
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".csv")]
    if len(names) != 1:
        raise RuntimeError("Upload exactly one inference CSV.")
    new_data = pd.read_csv(Path("/content") / names[0])
    missing = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
    if missing:
        raise ValueError(f"Inference CSV is missing required features: {missing}")
    X = new_data[FEATURE_COLUMNS].copy()
    active = finetuned_predictor or baseline_predictor
    pred = active.predict(X)
    proba = active.predict_proba(X)
    out = new_data.copy()
    out["prediction"] = pred.values
    for col in proba.columns:
        out[f"probability_{col}"] = proba[col].values
    out.to_csv("/content/predictions.csv", index=False)
    display(out.head())
    files.download("/content/predictions.csv")
else:
    print("Inference skipped.")


## 6. Export the reusable predictor

The AutoGluon `TabularPredictor` directory is the reusable trained artifact. The exported ZIP is not a replacement `model.safetensors`; extract it and load the directory with `TabularPredictor.load(path)`.


In [ ]:
active_predictor = finetuned_predictor or baseline_predictor
active_path = Path(active_predictor.path)
metadata = {
    "base_model": MODEL_ID,
    "base_model_revision": PINNED_REVISION,
    "weights_sha256": EXPECTED_WEIGHTS_SHA256,
    "config_sha256": EXPECTED_CONFIG_SHA256,
    "autogluon_version": "1.5.0",
    "mode": "fine-tuned" if finetuned_predictor is not None else "pretrained",
    "target_column": TARGET_COLUMN,
    "features": FEATURE_COLUMNS,
    "seed": SEED,
    "data_source": DATA_SOURCE,
    "sample_dataset_card": SAMPLE_CARD_URL if using_presplit_sample else None,
    "ai_assistance": {
        "client": "OpenAI ChatGPT",
        "agent_relay_role": "Builder",
        "note": "Attribution is provenance, not sign-off or independent verification.",
    },
}
(active_path / "tutorial_run_metadata.json").write_text(json.dumps(metadata, indent=2))
archive = shutil.make_archive("/content/mitra-predictor", "zip", root_dir=active_path)
print("✓ Predictor archive:", archive)


## AI use and provenance

This tutorial was developed with substantial AI assistance from **OpenAI ChatGPT** under human direction and review.

- Agent Relay role: **Builder**
- Base-model developer: **AutoGluon team, Amazon Web Services (AWS)**
- DIMER role: distributor of the pinned `model.safetensors` artifact, not model developer

AI attribution is **provenance, not sign-off** and does not independently verify correctness.
